# Blip2 Opt 2 7b COCO Pruned GA P30 Mr002

This notebook was reorganized for the GitHub reproducibility package.
Original file: `GA-I_P30_MR0.02/Blip-2 budamas#U0131[3, 6, 11]#L01f44c.ipynb_`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


# **BLIP-2 Prune Çalışmaları_2 İkinci Denemesi: EN İYİ SONUÇLARI ALDIĞIMIZ ÇALIŞMA**


> Düşük budama oranları ile yğksek CIDEr kayıpları yaşandı bu gözlem sonucunda tekrardan bir deneme yapıp sonuçları göreceğiz, bakalım nasıl olacak :)



In [ ]:
!apt-get install -y default-jdk -q
!pip install pymoo pycocotools pycocoevalcap -q
print('✅ Kurulum tamam')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# HÜCRE 3 — Tüm Verileri A100 Diskine Kopyala

import os, json, shutil, time
from concurrent.futures import ThreadPoolExecutor

COCO_DST   = '/content/images/coco_val2014'
NOCAPS_DST = '/content/images/nocaps_val'
PROXY_DST  = '/content/images/proxy_coco_200'
for d in [COCO_DST, NOCAPS_DST, PROXY_DST]:
    os.makedirs(d, exist_ok=True)

COCO_SRC       = '/content/drive/MyDrive/datasets/coco2014/val2014'
NOCAPS_SRC     = '/content/drive/MyDrive/datasets/nocaps/images_val_hf'
PROXY_SRC      = '/content/drive/MyDrive/proxy_coco_200/images'
TEST_JSON      = '/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json'
GT_JSON_NOCAPS = '/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json'

def copy_file(args):
    src, dst = args
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

def copy_with_progress(pairs, label):
    total, done, t0 = len(pairs), 0, time.time()
    with ThreadPoolExecutor(max_workers=16) as ex:
        for _ in ex.map(copy_file, pairs):
            done += 1
            if done % 200 == 0 or done == total:
                print(f'  {label}: {done}/{total} — {time.time()-t0:.0f} sn', end='\r')
    print(f'  {label}: {done}/{total} — tamamlandı ({time.time()-t0:.0f} sn)         ')

with open(TEST_JSON) as f:
    test_images = json.load(f)
if len(os.listdir(COCO_DST)) < len(test_images):
    print(f'COCO kopyalanıyor ({len(test_images)} görüntü)...')
    pairs = [(f"{COCO_SRC}/{item['image'].split('/')[-1]}",
              f"{COCO_DST}/{item['image'].split('/')[-1]}") for item in test_images]
    copy_with_progress(pairs, 'COCO')
else:
    print(f'✅ COCO zaten diskte ({len(os.listdir(COCO_DST))} dosya)')

with open(GT_JSON_NOCAPS) as f:
    nocaps_data = json.load(f)
nocaps_images_all = nocaps_data['images']
if len(os.listdir(NOCAPS_DST)) < len(nocaps_images_all):
    print(f'NoCaps kopyalanıyor ({len(nocaps_images_all)} görüntü)...')
    pairs = [(f"{NOCAPS_SRC}/{img['file_name']}",
              f"{NOCAPS_DST}/{img['file_name']}") for img in nocaps_images_all]
    copy_with_progress(pairs, 'NoCaps')
else:
    print(f'✅ NoCaps zaten diskte ({len(os.listdir(NOCAPS_DST))} dosya)')

proxy_files = os.listdir(PROXY_SRC)
if len(os.listdir(PROXY_DST)) < len(proxy_files):
    print(f'Proxy kopyalanıyor ({len(proxy_files)} görüntü)...')
    pairs = [(f'{PROXY_SRC}/{fn}', f'{PROXY_DST}/{fn}') for fn in proxy_files]
    copy_with_progress(pairs, 'Proxy')
else:
    print(f'✅ Proxy zaten diskte ({len(os.listdir(PROXY_DST))} dosya)')

print('\n' + '='*50)
print(f'✅ COCO  : {len(os.listdir(COCO_DST)):>5} dosya  → {COCO_DST}')
print(f'✅ NoCaps: {len(os.listdir(NOCAPS_DST)):>5} dosya  → {NOCAPS_DST}')
print(f'✅ Proxy : {len(os.listdir(PROXY_DST)):>5} dosya  → {PROXY_DST}')

In [ ]:
# Hücre - 4 Model Yükle + GA + Otomatik Seçim

import json, os, torch, time, copy
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from pycocoevalcap.cider.cider import Cider
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.core.callback import Callback

# ════════════════════════════════════════════════════════
# AYARLAR — LAVIS orijinal parametreleri
# ════════════════════════════════════════════════════════
MODEL_ID           = 'Salesforce/blip2-opt-2.7b'
DEVICE             = 'cuda'
PROMPT             = ''
NUM_BEAMS          = 5
MAX_NEW_TOKENS     = 30
MIN_NEW_TOKENS     = 8
REPETITION_PENALTY = 1.15
DO_SAMPLE          = False

N_BLOCKS         = 32
MAX_PRUNE        = 16   # %50
BATCH_SIZE_PROXY = 64

PROXY_JSON      = '/content/drive/MyDrive/proxy_coco_200/proxy_coco_200_annotations.json'
RESULTS_DIR     = '/content/drive/MyDrive/ga_results/blip2'
CHECKPOINT_PATH = os.path.join(RESULTS_DIR, 'ga_checkpoint_2.json')
RESULTS_PATH    = os.path.join(RESULTS_DIR, 'pareto_solutions_2.json')
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(PROXY_JSON) as f:
    proxy_data = json.load(f)
proxy_images_list = proxy_data['images']
print(f'✅ Proxy set yüklendi: {len(proxy_images_list)} görüntü')

print('Model yükleniyor...')
processor = Blip2Processor.from_pretrained(MODEL_ID)
base_model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto'
).eval()
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
original_params = sum(p.numel() for p in base_model.parameters())
print(f'✅ Model hazır | Parametre: {original_params:,} ({original_params/1e9:.3f} B)')

# ── eval.py patch (SPICE kaldır) ──
import inspect, pycocoevalcap
from pathlib import Path
import pycocoevalcap.eval as pe
eval_path = Path(inspect.getfile(pe))
txt = eval_path.read_text()
txt2 = txt.replace("from .spice.spice import Spice\n", "")
txt2 = txt2.replace("(Spice(), \"SPICE\")", "")
if txt2 != txt:
    eval_path.write_text(txt2)
    print("✅ eval.py patch edildi (SPICE kaldırıldı)")
else:
    print("✅ eval.py zaten patch edilmiş")

def repair(chromosome, max_prune):
    chromosome = chromosome.copy()
    zero_idx = [i for i, b in enumerate(chromosome) if b == 0]
    if len(zero_idx) > max_prune:
        restore = np.random.choice(zero_idx, len(zero_idx) - max_prune, replace=False)
        for idx in restore:
            chromosome[idx] = 1
    return chromosome

def prune_model(model, chromosome):
    pruned = copy.deepcopy(model)
    layers = pruned.language_model.model.decoder.layers
    keep   = [i for i, b in enumerate(chromosome) if b == 1]
    pruned.language_model.model.decoder.layers = torch.nn.ModuleList([layers[i] for i in keep])
    for layer in pruned.language_model.model.decoder.layers:
        for param in layer.parameters():
            if not param.is_contiguous():
                param.data = param.data.contiguous()
    pruned_params   = sum(p.numel() for p in pruned.parameters())
    param_drop_rate = (original_params - pruned_params) / original_params
    return pruned, param_drop_rate

def run_inference_proxy(model, proxy_images, batch_size):
    model.eval()
    results = {}
    for i in tqdm(range(0, len(proxy_images), batch_size), desc='Inference'):
        batch = proxy_images[i:i+batch_size]
        imgs, ids = [], []
        for item in batch:
            imgs.append(Image.open(os.path.join(PROXY_DST, item['filename'])).convert('RGB'))
            ids.append(item['image_id'])
        inputs = processor(
            images=imgs,
            return_tensors='pt'
        ).to(DEVICE, torch.float16)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                do_sample=DO_SAMPLE,
                num_beams=NUM_BEAMS,
                max_new_tokens=MAX_NEW_TOKENS,
                min_new_tokens=MIN_NEW_TOKENS,
                repetition_penalty=REPETITION_PENALTY,
            )
        captions = processor.batch_decode(out, skip_special_tokens=True)
        for img_id, cap in zip(ids, captions):
            results[str(img_id)] = [cap.strip()]
    return results

def compute_cider_proxy(results):
    gts = {str(item['image_id']): item['captions'] for item in proxy_images_list}
    score, _ = Cider().compute_score(gts, results)
    return score

def run_inference_eval(model, images, img_dir, id_fn, batch_size):
    results = []
    for i in tqdm(range(0, len(images), batch_size), desc='Inference'):
        batch = images[i:i+batch_size]
        pil_imgs, img_ids = [], []
        for img_info in batch:
            fname, img_id = id_fn(img_info)
            pil_imgs.append(Image.open(os.path.join(img_dir, fname)).convert('RGB'))
            img_ids.append(img_id)
        inputs = processor(
            images=pil_imgs,
            return_tensors='pt'
        ).to(DEVICE, torch.float16)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                do_sample=DO_SAMPLE,
                num_beams=NUM_BEAMS,
                max_new_tokens=MAX_NEW_TOKENS,
                min_new_tokens=MIN_NEW_TOKENS,
                repetition_penalty=REPETITION_PENALTY,
            )
        captions = processor.batch_decode(out, skip_special_tokens=True)
        for img_id, cap in zip(img_ids, captions):
            results.append({'image_id': img_id, 'caption': cap.strip()})
        done = min(i + batch_size, len(images))
        if (i // batch_size) % 20 == 0:
            print(f'  {done}/{len(images)} — Örnek caption: {captions[0][:70]}')
    return results

def compute_metrics(results, gt_json, save_dir, label, dataset):
    import sys
    from pathlib import Path
    import inspect

    # Her çağrıda modülü temizle ve patch'i garantile
    for key in list(sys.modules.keys()):
        if 'pycocoevalcap' in key:
            del sys.modules[key]

    import pycocoevalcap.eval as pe
    eval_path = Path(inspect.getfile(pe))
    txt = eval_path.read_text()
    txt2 = txt.replace("from .spice.spice import Spice\n", "")
    txt2 = txt2.replace("(Spice(), \"SPICE\")", "")
    if txt2 != txt:
        eval_path.write_text(txt2)

    from pycocotools.coco import COCO
    from pycocoevalcap.eval import COCOEvalCap

    pred_path = os.path.join(save_dir, f'blip2_{dataset}_{label}_preds_2.json')
    with open(pred_path, 'w') as f:
        json.dump(results, f)

    coco      = COCO(gt_json)
    coco_res  = coco.loadRes(pred_path)
    coco_eval = COCOEvalCap(coco, coco_res)
    coco_eval.params['image_id'] = [r['image_id'] for r in results]

    try:
        coco_eval.evaluate()
    except Exception as e:
        print(f'  ⚠️ Eval hatası (görmezden gelindi): {e}')

    return coco_eval.eval

class PruningProblem(Problem):
    def __init__(self):
        super().__init__(n_var=N_BLOCKS, n_obj=2, n_constr=0,
                         xl=0, xu=1, vtype=bool)
    def _evaluate(self, X, out, *args, **kwargs):
        F = []
        for x in X:
            chrom = repair(x.tolist(), MAX_PRUNE)
            pruned, drop_rate = prune_model(base_model, chrom)
            results = run_inference_proxy(pruned, proxy_images_list, BATCH_SIZE_PROXY)
            cider   = compute_cider_proxy(results)
            F.append([-drop_rate, -cider])
            del pruned
            torch.cuda.empty_cache()
        out['F'] = np.array(F)

class EarlyStoppingCallback(Callback):
    def __init__(self, patience=20):
        super().__init__()
        self.patience   = patience
        self.best_cider = -np.inf
        self.counter    = 0
        if os.path.exists(CHECKPOINT_PATH):
            with open(CHECKPOINT_PATH) as f:
                ckpt = json.load(f)
            self.best_cider = ckpt.get('best_cider', -np.inf)
            self.counter    = ckpt.get('patience_counter', 0)
            print(f'✅ Checkpoint yüklendi — best CIDEr: {self.best_cider:.4f} | İyileşme yok: {self.counter}/{self.patience}')

    def notify(self, algorithm):
        F   = algorithm.pop.get('F')
        gen = algorithm.n_gen
        best_cider = float(np.max(-F[:, 1]))
        if best_cider > self.best_cider + 1e-6:
            self.best_cider = best_cider
            self.counter    = 0
        else:
            self.counter += 1
        pareto_ckpt = []
        for f, x in zip(F, algorithm.pop.get('X')):
            chrom = repair(x.tolist(), MAX_PRUNE)
            pareto_ckpt.append({
                'chromosome': chrom,
                'param_drop_rate': float(-f[0]),
                'cider_score': float(-f[1])
            })
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump({'last_gen': gen, 'best_cider': self.best_cider,
                       'patience_counter': self.counter,
                       'pareto_solutions': pareto_ckpt}, f)
        print(f'Nesil {gen:3d} | En iyi CIDEr: {best_cider:.4f} | İyileşme yok: {self.counter}/{self.patience}')
        if self.counter >= self.patience:
            print('⛔ Erken durdurma tetiklendi!')
            algorithm.termination.force_termination = True

algorithm = NSGA2(
    pop_size=30,
    sampling=BinaryRandomSampling(),
    crossover=SBX(prob=0.9),
    mutation=BitflipMutation(prob=0.02),
    eliminate_duplicates=True,
)
termination = get_termination('n_gen', 50)

print(f'\n🚀 GA başlıyor...')
print(f'   Model: {MODEL_ID} | Blok: {N_BLOCKS} | Pop: 30 | Nesil: 50 | Max budama: {MAX_PRUNE}')
print(f'   NUM_BEAMS={NUM_BEAMS} | MAX_NEW_TOKENS={MAX_NEW_TOKENS} | MIN_NEW_TOKENS={MIN_NEW_TOKENS}')
print(f'   REPETITION_PENALTY={REPETITION_PENALTY} | PROMPT="{PROMPT}"')
print('-' * 60)

t0  = time.time()
res = minimize(
    PruningProblem(), algorithm, termination,
    callback=EarlyStoppingCallback(patience=20),
    seed=42, verbose=False
)
elapsed = time.time() - t0
print('-' * 60)
print(f'✅ GA tamamlandı! Süre: {int(elapsed//3600)}s {int((elapsed%3600)//60)}dk {int(elapsed%60)}sn')
print(f'   Pareto cephesi: {len(res.F)} çözüm')

pareto_solutions = []
for i, (f, x) in enumerate(zip(res.F, res.X)):
    chrom = repair(x.tolist(), MAX_PRUNE)
    pareto_solutions.append({
        'index'          : i,
        'chromosome'     : chrom,
        'param_drop_rate': float(-f[0]),
        'cider_score'    : float(-f[1])
    })

with open(RESULTS_PATH, 'w') as f:
    json.dump({'model_id': MODEL_ID, 'n_blocks': N_BLOCKS,
               'max_prune': MAX_PRUNE, 'original_params': original_params,
               'solutions': pareto_solutions}, f, indent=2)
print(f'✅ Kaydedildi: {RESULTS_PATH}')

# ── Pareto tablosu ──
print('\nPareto Cephesi:')
print(f"  {'Çözüm':>6} | {'Param Düşüşü':>12} | {'CIDEr':>8}")
print('-' * 38)
for sol in sorted(pareto_solutions, key=lambda x: x['param_drop_rate']):
    print(f"  {sol['index']:>6} | %{sol['param_drop_rate']*100:>10.1f} | {sol['cider_score']:>8.4f}")

# ── Otomatik 4 nokta seç: param_drop_rate'e göre sırala, 4 eşit dilimden 1'er al ──
sorted_sols = sorted(pareto_solutions, key=lambda x: x['param_drop_rate'])
n = len(sorted_sols)
label_names = ['low', 'mid', 'high', 'extra_high']

if n >= 4:
    picks = [int(n * 0.15), int(n * 0.38), int(n * 0.62), int(n * 0.85)]
    picks = [min(p, n-1) for p in picks]
    seen, clean = set(), []
    for p in picks:
        while p in seen and p < n-1:
            p += 1
        seen.add(p)
        clean.append(p)
    selected_solutions = [sorted_sols[i] for i in clean]
else:
    selected_solutions = sorted_sols[:4]

label_map = {sol['index']: label_names[i] for i, sol in enumerate(selected_solutions)}
SELECTED  = [sol['index'] for sol in selected_solutions]

print('\n✅ Otomatik seçilen 4 Pareto noktası:')
for sol in selected_solutions:
    lbl = label_map[sol['index']]
    print(f'  [{lbl.upper():>10}] index:{sol["index"]} | %{sol["param_drop_rate"]*100:.1f} | proxy CIDEr:{sol["cider_score"]:.4f}')

if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print('✅ Checkpoint temizlendi')

In [ ]:
# Hücre-5 Pareto Grafiği

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASELINE = 1.335

sorted_plot = sorted(pareto_solutions, key=lambda x: x['param_drop_rate'])
px = [s['param_drop_rate'] * 100 for s in sorted_plot]
py = [s['cider_score'] for s in sorted_plot]

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#1e293b')
ax.grid(color='#334155', linestyle='--', linewidth=0.6, alpha=0.7)
ax.axhline(y=BASELINE, color='#f59e0b', linestyle='--', linewidth=1.5,
           label=f'Baseline CIDEr = {BASELINE}')
ax.plot(px, py, color='#475569', linestyle='--', linewidth=1.2, zorder=2)
ax.scatter(px, py, color='#38bdf8', s=90, zorder=3,
           label='Pareto noktaları (proxy, 200 görüntü)')

for s in sorted_plot:
    ax.annotate(
        f"%{s['param_drop_rate']*100:.1f}\n{s['cider_score']:.4f}",
        (s['param_drop_rate']*100, s['cider_score']),
        textcoords='offset points', xytext=(0, 12),
        ha='center', fontsize=8, color='#e2e8f0'
    )

for sol in selected_solutions:
    lbl = label_map[sol['index']]
    ax.scatter(sol['param_drop_rate']*100, sol['cider_score'],
               color='#f472b6', s=160, zorder=5, marker='*')
    ax.annotate(f'← {lbl}',
                (sol['param_drop_rate']*100, sol['cider_score']),
                textcoords='offset points', xytext=(8, -4),
                fontsize=8, color='#f472b6')

ax.set_xlabel('Parametre Düşüşü (%)', color='#94a3b8', fontsize=12)
ax.set_ylabel('Proxy CIDEr (200 görüntü)', color='#94a3b8', fontsize=12)
ax.set_title('BLIP-2 — NSGA-II Pareto Cephesi', color='#e2e8f0', fontsize=14)
ax.tick_params(colors='#94a3b8')
ax.legend(facecolor='#1e293b', labelcolor='#e2e8f0', fontsize=8)
plt.tight_layout()

PLOT_PATH = '/content/drive/MyDrive/ga_results/blip2/blip2_pareto_proxy_2.png'
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Grafik kaydedildi: {PLOT_PATH}')

In [ ]:
# HÜCRE 6 — COCO Final Eval

import os, json, torch, copy, time, re
from PIL import Image
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap

PROMPT             = ''
NUM_BEAMS          = 5
MAX_NEW_TOKENS     = 30
MIN_NEW_TOKENS     = 8
REPETITION_PENALTY = 1.15
DO_SAMPLE          = False
BATCH_SIZE_EVAL    = 64
DEVICE             = 'cuda'

GT_JSON   = '/content/drive/MyDrive/coco_karpathy/coco_karpathy_test_gt.json'
TEST_JSON = '/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json'
SAVE_DIR  = '/content/drive/MyDrive/ga_results/blip2/final_eval_2'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(TEST_JSON) as f:
    test_images = json.load(f)
print(f'COCO test seti: {len(test_images)} görüntü')
print(f'\nKullanılacak parametreler:')
print(f'  PROMPT="{PROMPT}" | NUM_BEAMS={NUM_BEAMS} | MAX_NEW_TOKENS={MAX_NEW_TOKENS}')
print(f'  MIN_NEW_TOKENS={MIN_NEW_TOKENS} | REPETITION_PENALTY={REPETITION_PENALTY}')
print(f'\nSeçilen Pareto çözümleri:')
for sol in selected_solutions:
    lbl = label_map[sol['index']]
    print(f'  [{lbl.upper():>10}] index:{sol["index"]} | %{sol["param_drop_rate"]*100:.1f} | proxy CIDEr:{sol["cider_score"]:.4f}')

def coco_id_fn(img_info):
    fname  = img_info['image'].split('/')[-1]
    img_id = int(re.search(r'_(\d{12})\.jpg$', fname).group(1))
    return fname, img_id

summary = []
t_total = time.time()

for sol in selected_solutions:
    label    = label_map[sol['index']]
    n_kept   = sum(sol['chromosome'])
    n_pruned = len(sol['chromosome']) - n_kept
    print(f"\n{'='*50}")
    print(f"[{label.upper()}] index:{sol['index']} | %{sol['param_drop_rate']*100:.1f} düşüş | proxy CIDEr:{sol['cider_score']:.4f}")
    print(f'  {n_pruned} blok silindi, {n_kept} kaldı')

    model   = prune_model(base_model, sol['chromosome'])[0]
    t0      = time.time()
    results = run_inference_eval(model, test_images, COCO_DST, coco_id_fn, BATCH_SIZE_EVAL)
    metrics = compute_metrics(results, GT_JSON, SAVE_DIR, label, 'coco')

    cider  = metrics.get('CIDEr', 0)
    bleu4  = metrics.get('Bleu_4', 0)
    meteor = metrics.get('METEOR', 0)
    rouge  = metrics.get('ROUGE_L', 0)
    print(f'  CIDEr : {cider:.4f}')
    print(f'  BLEU-4: {bleu4:.4f}')
    print(f'  METEOR: {meteor:.4f}')
    print(f'  ROUGE-L:{rouge:.4f}')
    print(f'  Süre  : {(time.time()-t0)/60:.1f} dk')

    entry = {
        'label': label, 'index': sol['index'],
        'param_drop_rate': sol['param_drop_rate'],
        'proxy_cider': sol['cider_score'],
        'final_cider': cider, 'final_bleu4': bleu4,
        'final_meteor': meteor, 'final_rouge_l': rouge,
        'chromosome': sol['chromosome']
    }
    summary.append(entry)
    out_path = os.path.join(SAVE_DIR, f'blip2_coco_{label}_eval_2.json')
    with open(out_path, 'w') as f:
        json.dump(entry, f, indent=2)
    print(f'  ✅ Kaydedildi: {out_path}')

    del model
    torch.cuda.empty_cache()

print(f'\n{"="*50}')
print(f'✅ COCO tamamlandı — toplam süre: {(time.time()-t_total)/60:.1f} dk')
print(f'\nÖZET:')
print(f"{'Label':<12} {'Param Düşüş':>12}   {'Proxy CIDEr':>12}   {'Final CIDEr':>12}    {'BLEU-4':>8}")
print('-' * 65)
for e in summary:
    print(f"{e['label']:<12} %{e['param_drop_rate']*100:>10.1f}   {e['proxy_cider']:>12.4f}   {e['final_cider']:>12.4f}    {e['final_bleu4']:>8.4f}")

with open(os.path.join(SAVE_DIR, 'blip2_coco_summary_2.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✅ Özet kaydedildi.')

In [ ]:
# HÜCRE 7 — NoCaps Final Eval

import os, json, torch, copy, time
from PIL import Image
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap

GT_JSON_NOCAPS  = '/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json'
SAVE_DIR_NOCAPS = '/content/drive/MyDrive/ga_results/blip2/final_eval_nocaps_2'
os.makedirs(SAVE_DIR_NOCAPS, exist_ok=True)

with open(GT_JSON_NOCAPS) as f:
    nocaps_data = json.load(f)
nocaps_images = nocaps_data['images']
print(f'NoCaps val seti: {len(nocaps_images)} görüntü')

def nocaps_id_fn(img_info):
    return img_info['file_name'], img_info['id']

summary_nocaps = []
t_total = time.time()

for sol in selected_solutions:
    label    = label_map[sol['index']]
    n_kept   = sum(sol['chromosome'])
    n_pruned = len(sol['chromosome']) - n_kept
    print(f"\n{'='*50}")
    print(f"[{label.upper()}] index:{sol['index']} | %{sol['param_drop_rate']*100:.1f} düşüş | proxy CIDEr:{sol['cider_score']:.4f}")
    print(f'  {n_pruned} blok silindi, {n_kept} kaldı')

    model   = prune_model(base_model, sol['chromosome'])[0]
    t0      = time.time()
    results = run_inference_eval(model, nocaps_images, NOCAPS_DST, nocaps_id_fn, BATCH_SIZE_EVAL)
    metrics = compute_metrics(results, GT_JSON_NOCAPS, SAVE_DIR_NOCAPS, label, 'nocaps')

    cider  = metrics.get('CIDEr', 0)
    bleu4  = metrics.get('Bleu_4', 0)
    meteor = metrics.get('METEOR', 0)
    rouge  = metrics.get('ROUGE_L', 0)
    print(f'  CIDEr : {cider:.4f}')
    print(f'  BLEU-4: {bleu4:.4f}')
    print(f'  METEOR: {meteor:.4f}')
    print(f'  ROUGE-L:{rouge:.4f}')
    print(f'  Süre  : {(time.time()-t0)/60:.1f} dk')

    entry = {
        'label': label, 'index': sol['index'],
        'param_drop_rate': sol['param_drop_rate'],
        'proxy_cider': sol['cider_score'],
        'final_cider': cider, 'final_bleu4': bleu4,
        'final_meteor': meteor, 'final_rouge_l': rouge,
        'chromosome': sol['chromosome']
    }
    summary_nocaps.append(entry)
    out_path = os.path.join(SAVE_DIR_NOCAPS, f'blip2_nocaps_{label}_eval_2.json')
    with open(out_path, 'w') as f:
        json.dump(entry, f, indent=2)
    print(f'  ✅ Kaydedildi: {out_path}')

    del model
    torch.cuda.empty_cache()

print(f'\n{"="*50}')
print(f'✅ NoCaps tamamlandı — toplam süre: {(time.time()-t_total)/60:.1f} dk')
print(f'\nÖZET:')
print(f"{'Label':<12} {'Param Düşüş':>12}   {'Proxy CIDEr':>12}   {'Final CIDEr':>12}    {'BLEU-4':>8}")
print('-' * 65)
for e in summary_nocaps:
    print(f"{e['label']:<12} %{e['param_drop_rate']*100:>10.1f}   {e['proxy_cider']:>12.4f}   {e['final_cider']:>12.4f}    {e['final_bleu4']:>8.4f}")

with open(os.path.join(SAVE_DIR_NOCAPS, 'blip2_nocaps_summary_2.json'), 'w') as f:
    json.dump(summary_nocaps, f, indent=2)
print(f'✅ Özet kaydedildi.')

#**İlk 11 Merakı İçin Farklı Bir Deneme** ⚡⚡⚡

In [ ]:
!apt-get install -y default-jdk -q
!pip install pymoo pycocotools pycocoevalcap -q
print('✅ Kurulum tamam')

In [ ]:
import json, os, torch, time, copy, re, sys
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from google.colab import drive
drive.mount('/content/drive')

MODEL_ID           = 'Salesforce/blip2-opt-2.7b'
DEVICE             = 'cuda'
PROMPT             = ''
NUM_BEAMS          = 5
MAX_NEW_TOKENS     = 30
MIN_NEW_TOKENS     = 8
REPETITION_PENALTY = 1.15
DO_SAMPLE          = False
BATCH_SIZE_EVAL    = 64

COCO_DST   = '/content/images/coco_val2014'
NOCAPS_DST = '/content/images/nocaps_val'

print('Model yükleniyor...')
processor = Blip2Processor.from_pretrained(MODEL_ID)
base_model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto'
).eval()
original_params = sum(p.numel() for p in base_model.parameters())
print(f'✅ Model hazır | {original_params:,} ({original_params/1e9:.3f} B)')

# eval.py patch
from pathlib import Path
import inspect, pycocoevalcap.eval as pe
eval_path = Path(inspect.getfile(pe))
txt = eval_path.read_text()
txt2 = txt.replace("from .spice.spice import Spice\n", "")
txt2 = txt2.replace("(Spice(), \"SPICE\")", "")
if txt2 != txt:
    eval_path.write_text(txt2)
    print("✅ eval.py patch edildi")
else:
    print("✅ eval.py zaten patch edilmiş")

# Pareto çözümlerini yükle
RESULTS_PATH = '/content/drive/MyDrive/ga_results/blip2/pareto_solutions_2.json'
with open(RESULTS_PATH) as f:
    pareto_data = json.load(f)
pareto_solutions = pareto_data['solutions']

print('\nPareto Cephesi:')
print(f"  {'Çözüm':>6} | {'Param Düşüşü':>12} | {'CIDEr':>8}")
print('-' * 38)
for sol in sorted(pareto_solutions, key=lambda x: x['param_drop_rate']):
    print(f"  {sol['index']:>6} | %{sol['param_drop_rate']*100:>10.1f} | {sol['cider_score']:>8.4f}")

def repair(chromosome, max_prune=16):
    chromosome = chromosome.copy()
    zero_idx = [i for i, b in enumerate(chromosome) if b == 0]
    if len(zero_idx) > max_prune:
        restore = np.random.choice(zero_idx, len(zero_idx) - max_prune, replace=False)
        for idx in restore:
            chromosome[idx] = 1
    return chromosome

def prune_model(model, chromosome):
    pruned = copy.deepcopy(model)
    layers = pruned.language_model.model.decoder.layers
    keep   = [i for i, b in enumerate(chromosome) if b == 1]
    pruned.language_model.model.decoder.layers = torch.nn.ModuleList([layers[i] for i in keep])
    for layer in pruned.language_model.model.decoder.layers:
        for param in layer.parameters():
            if not param.is_contiguous():
                param.data = param.data.contiguous()
    pruned_params   = sum(p.numel() for p in pruned.parameters())
    param_drop_rate = (original_params - pruned_params) / original_params
    return pruned, param_drop_rate

def run_inference_eval(model, images, img_dir, id_fn, batch_size):
    results = []
    for i in tqdm(range(0, len(images), batch_size), desc='Inference'):
        batch = images[i:i+batch_size]
        pil_imgs, img_ids = [], []
        for img_info in batch:
            fname, img_id = id_fn(img_info)
            pil_imgs.append(Image.open(os.path.join(img_dir, fname)).convert('RGB'))
            img_ids.append(img_id)
        inputs = processor(images=pil_imgs, return_tensors='pt').to(DEVICE, torch.float16)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                do_sample=DO_SAMPLE,
                num_beams=NUM_BEAMS,
                max_new_tokens=MAX_NEW_TOKENS,
                min_new_tokens=MIN_NEW_TOKENS,
                repetition_penalty=REPETITION_PENALTY,
            )
        captions = processor.batch_decode(out, skip_special_tokens=True)
        for img_id, cap in zip(img_ids, captions):
            results.append({'image_id': img_id, 'caption': cap.strip()})
        done = min(i + batch_size, len(images))
        if (i // batch_size) % 20 == 0:
            print(f'  {done}/{len(images)} — Örnek caption: {captions[0][:70]}')
    return results

def compute_metrics_merak(results, gt_json, save_dir, label, dataset):
    for key in list(sys.modules.keys()):
        if 'pycocoevalcap' in key:
            del sys.modules[key]
    import pycocoevalcap.eval as pe
    from pathlib import Path
    import inspect
    eval_path = Path(inspect.getfile(pe))
    txt = eval_path.read_text()
    txt2 = txt.replace("from .spice.spice import Spice\n", "")
    txt2 = txt2.replace("(Spice(), \"SPICE\")", "")
    if txt2 != txt:
        eval_path.write_text(txt2)
    from pycocotools.coco import COCO
    from pycocoevalcap.eval import COCOEvalCap
    pred_path = os.path.join(save_dir, f'blip2_{dataset}_{label}_merak.json')
    with open(pred_path, 'w') as f:
        json.dump(results, f)
    coco      = COCO(gt_json)
    coco_res  = coco.loadRes(pred_path)
    coco_eval = COCOEvalCap(coco, coco_res)
    coco_eval.params['image_id'] = [r['image_id'] for r in results]
    try:
        coco_eval.evaluate()
    except Exception as e:
        print(f'  ⚠️ Eval hatası (görmezden gelindi): {e}')
    return coco_eval.eval

print('\n✅ Hazır!')

In [ ]:
import shutil
from concurrent.futures import ThreadPoolExecutor

COCO_SRC   = '/content/drive/MyDrive/datasets/coco2014/val2014'
NOCAPS_SRC = '/content/drive/MyDrive/datasets/nocaps/images_val_hf'
TEST_JSON  = '/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json'
GT_JSON_NOCAPS = '/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json'

os.makedirs(COCO_DST, exist_ok=True)
os.makedirs(NOCAPS_DST, exist_ok=True)

def copy_file(args):
    src, dst = args
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

def copy_with_progress(pairs, label):
    total, done, t0 = len(pairs), 0, time.time()
    with ThreadPoolExecutor(max_workers=16) as ex:
        for _ in ex.map(copy_file, pairs):
            done += 1
            if done % 200 == 0 or done == total:
                print(f'  {label}: {done}/{total} — {time.time()-t0:.0f} sn', end='\r')
    print(f'  {label}: {done}/{total} — tamamlandı ({time.time()-t0:.0f} sn)         ')

with open(TEST_JSON) as f:
    test_images = json.load(f)
if len(os.listdir(COCO_DST)) < len(test_images):
    print(f'COCO kopyalanıyor...')
    pairs = [(f"{COCO_SRC}/{item['image'].split('/')[-1]}",
              f"{COCO_DST}/{item['image'].split('/')[-1]}") for item in test_images]
    copy_with_progress(pairs, 'COCO')
else:
    print(f'✅ COCO zaten diskte ({len(os.listdir(COCO_DST))} dosya)')

with open(GT_JSON_NOCAPS) as f:
    nocaps_data = json.load(f)
nocaps_images = nocaps_data['images']
if len(os.listdir(NOCAPS_DST)) < len(nocaps_images):
    print(f'NoCaps kopyalanıyor...')
    pairs = [(f"{NOCAPS_SRC}/{img['file_name']}",
              f"{NOCAPS_DST}/{img['file_name']}") for img in nocaps_images]
    copy_with_progress(pairs, 'NoCaps')
else:
    print(f'✅ NoCaps zaten diskte ({len(os.listdir(NOCAPS_DST))} dosya)')

print('✅ Görüntüler hazır!')

In [ ]:
# ── Merak: İlk 11 indeksi değerlendir ──
SAVE_DIR_MERAK = '/content/drive/MyDrive/ga_results/blip2/merak'
os.makedirs(SAVE_DIR_MERAK, exist_ok=True)

GT_JSON        = '/content/drive/MyDrive/coco_karpathy/coco_karpathy_test_gt.json'
TEST_JSON      = '/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json'
GT_JSON_NOCAPS = '/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json'

with open(TEST_JSON) as f:
    test_images = json.load(f)
with open(GT_JSON_NOCAPS) as f:
    nocaps_data = json.load(f)
nocaps_images = nocaps_data['images']

# İlk 11 indeks: param_drop_rate'e göre sırala, ilk 11'i al
sorted_sols = sorted(pareto_solutions, key=lambda x: x['param_drop_rate'])
merak_solutions = sorted_sols[:11]

def coco_id_fn(img_info):
    fname  = img_info['image'].split('/')[-1]
    img_id = int(re.search(r'_(\d{12})\.jpg$', fname).group(1))
    return fname, img_id

def nocaps_id_fn(img_info):
    return img_info['file_name'], img_info['id']

summary_coco   = []
summary_nocaps = []
t_total = time.time()

print(f'Toplam {len(merak_solutions)} Pareto noktası değerlendirilecek')
print(f'Sonuçlar: {SAVE_DIR_MERAK}\n')

for sol in merak_solutions:
    idx   = sol['index']
    drop  = sol['param_drop_rate'] * 100
    n_kept   = sum(sol['chromosome'])
    n_pruned = len(sol['chromosome']) - n_kept

    print(f"\n{'='*55}")
    print(f"index:{idx} | %{drop:.1f} düşüş | proxy CIDEr:{sol['cider_score']:.4f}")
    print(f"  {n_pruned} blok silindi, {n_kept} kaldı")

    model = prune_model(base_model, sol['chromosome'])[0]

    # ── COCO ──
    print(f"  → COCO eval...")
    t0      = time.time()
    results = run_inference_eval(model, test_images, COCO_DST, coco_id_fn, BATCH_SIZE_EVAL)
    metrics = compute_metrics_merak(results, GT_JSON, SAVE_DIR_MERAK, f'idx{idx}', 'coco')
    cider_c = metrics.get('CIDEr', 0)
    bleu4_c = metrics.get('Bleu_4', 0)
    print(f"  COCO  — CIDEr: {cider_c:.4f} | BLEU-4: {bleu4_c:.4f} | Süre: {(time.time()-t0)/60:.1f} dk")
    summary_coco.append({'index': idx, 'param_drop': drop,
                         'proxy_cider': sol['cider_score'],
                         'coco_cider': cider_c, 'coco_bleu4': bleu4_c})

    # ── NoCaps ──
    print(f"  → NoCaps eval...")
    t0      = time.time()
    results = run_inference_eval(model, nocaps_images, NOCAPS_DST, nocaps_id_fn, BATCH_SIZE_EVAL)
    metrics = compute_metrics_merak(results, GT_JSON_NOCAPS, SAVE_DIR_MERAK, f'idx{idx}', 'nocaps')
    cider_n = metrics.get('CIDEr', 0)
    bleu4_n = metrics.get('Bleu_4', 0)
    print(f"  NoCaps— CIDEr: {cider_n:.4f} | BLEU-4: {bleu4_n:.4f} | Süre: {(time.time()-t0)/60:.1f} dk")
    summary_nocaps.append({'index': idx, 'param_drop': drop,
                           'proxy_cider': sol['cider_score'],
                           'nocaps_cider': cider_n, 'nocaps_bleu4': bleu4_n})

    del model
    torch.cuda.empty_cache()

# ── Özet ──
print(f'\n{"="*55}')
print(f'✅ Tamamlandı — toplam süre: {(time.time()-t_total)/60:.1f} dk')

print(f'\nCOCO ÖZET (baseline: 1.335):')
print(f"  {'idx':>4} | {'Param%':>7} | {'Proxy':>7} | {'COCO CIDEr':>10} | {'BLEU-4':>7}")
print('-' * 48)
for e in summary_coco:
    print(f"  {e['index']:>4} | %{e['param_drop']:>5.1f} | {e['proxy_cider']:>7.4f} | {e['coco_cider']:>10.4f} | {e['coco_bleu4']:>7.4f}")

print(f'\nNoCaps ÖZET (baseline: 1.119):')
print(f"  {'idx':>4} | {'Param%':>7} | {'Proxy':>7} | {'NoCaps CIDEr':>12} | {'BLEU-4':>7}")
print('-' * 50)
for e in summary_nocaps:
    print(f"  {e['index']:>4} | %{e['param_drop']:>5.1f} | {e['proxy_cider']:>7.4f} | {e['nocaps_cider']:>12.4f} | {e['nocaps_bleu4']:>7.4f}")

# Drive'a kaydet
with open(os.path.join(SAVE_DIR_MERAK, 'merak_coco_summary.json'), 'w') as f:
    json.dump(summary_coco, f, indent=2)
with open(os.path.join(SAVE_DIR_MERAK, 'merak_nocaps_summary.json'), 'w') as f:
    json.dump(summary_nocaps, f, indent=2)
print(f'\n✅ Sonuçlar kaydedildi: {SAVE_DIR_MERAK}')

In [ ]:
# Grafikler

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASELINE_COCO   = 1.335
BASELINE_NOCAPS = 1.119
PLOT_DIR        = '/content/drive/MyDrive/ga_results/blip2/merak'

# ── Veriyi hazırla ──
x_coco   = [e['param_drop'] for e in summary_coco]
y_coco   = [e['coco_cider'] for e in summary_coco]
p_coco   = [e['proxy_cider'] for e in summary_coco]

x_nocaps = [e['param_drop'] for e in summary_nocaps]
y_nocaps = [e['nocaps_cider'] for e in summary_nocaps]
p_nocaps = [e['proxy_cider'] for e in summary_nocaps]

def make_plot(x, y_final, y_proxy, baseline, title, save_path, dataset_label):
    fig, ax1 = plt.subplots(figsize=(13, 7))
    fig.patch.set_facecolor('#0f172a')
    ax1.set_facecolor('#1e293b')
    ax1.grid(color='#334155', linestyle='--', linewidth=0.6, alpha=0.7)

    # Baseline
    ax1.axhline(y=baseline, color='#f59e0b', linestyle='--', linewidth=1.5,
                label=f'Baseline CIDEr = {baseline}')

    # Proxy CIDEr
    ax1.plot(x, y_proxy, color='#475569', linestyle=':', linewidth=1.2,
             marker='o', markersize=5, zorder=2, label='Proxy CIDEr (200 görüntü)')

    # Final CIDEr
    ax1.plot(x, y_final, color='#38bdf8', linestyle='-', linewidth=2,
             marker='o', markersize=8, zorder=3, label=f'Final CIDEr ({dataset_label})')

    # Annotate
    for xi, yf, yp, e in zip(x, y_final, y_proxy, summary_coco if 'COCO' in title else summary_nocaps):
        ax1.annotate(
            f"idx:{e['index']}\n{yf:.4f}",
            (xi, yf),
            textcoords='offset points', xytext=(0, 10),
            ha='center', fontsize=7, color='#e2e8f0'
        )

    ax1.set_xlabel('Parametre Düşüşü (%)', color='#94a3b8', fontsize=12)
    ax1.set_ylabel('CIDEr', color='#94a3b8', fontsize=12)
    ax1.set_title(title, color='#e2e8f0', fontsize=14)
    ax1.tick_params(colors='#94a3b8')
    ax1.legend(facecolor='#1e293b', labelcolor='#e2e8f0', fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Grafik kaydedildi: {save_path}')
    plt.close()

# ── COCO Grafiği ──
make_plot(
    x_coco, y_coco, p_coco,
    baseline  = BASELINE_COCO,
    title     = 'BLIP-2 — Tüm Pareto Noktaları | COCO Karpathy Test (Merak)',
    save_path = f'{PLOT_DIR}/merak_coco_plot.png',
    dataset_label = '5K COCO'
)

# ── NoCaps Grafiği ──
make_plot(
    x_nocaps, y_nocaps, p_nocaps,
    baseline  = BASELINE_NOCAPS,
    title     = 'BLIP-2 — Tüm Pareto Noktaları | NoCaps Val (Merak)',
    save_path = f'{PLOT_DIR}/merak_nocaps_plot.png',
    dataset_label = '4.5K NoCaps'
)